# The flat periodic sheet under three reservoirs and four fluidities

The matrix of `docs/bead_push_benchmark.md` section 9 without the bead, at the
code's default target tension of **0 pN/nm**: the elastic box, the global
relaxed area and the per-face relaxed area, each with the mesh frozen and with
edge flips at $\nu$ = 0.05, 0.5 and 5 per edge per µs, plus the fixed box at
$\nu$ = 0.5 as the no-reservoir control. 13 runs of 600 µs, frames every 0.2 µs
(`analysis/sticky_bead_runs.py --matrix --potential none --steps 600000`).

Three questions, one section each:

1. **Mean tension.** Does each reservoir hold the sheet at its target, and what
   does the box's reading decompose into?
2. **Space.** The height spectrum $\langle|h_q|^2\rangle$ on the limit surface,
   read through the subdivision route of
   `membrane_fluctuation_fluid_cpu.ipynb`, and the $\kappa_c$ and $\sigma$ it
   fits. Does the reservoir or the fluidity change the sheet the spectrum sees?
3. **Time.** The relaxation rate $\Gamma(q)$ of every mode from its
   autocorrelation, against the free-draining prediction
   $\Gamma = (D\,a/k_BT)\,(\kappa_c q^4 + \sigma q^2)$, with $a$ the area per vertex.
   Do flips or a reservoir change how fast the sheet moves?

The animations (section 6) are written beside the runs and are not tracked.
The runs are read from `SLIMED_FLAT_RUNS`, by default
`~/slimed_runs/flat_tension/matrix`. Nothing is simulated here.

In [ ]:
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import Image, Markdown, display


def find_repo_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for d in (here, *here.parents):
        if (d / "CMakeLists.txt").is_file() and (d / "analysis").is_dir():
            return d
    raise RuntimeError(f"no SLIMED checkout at or above {here}")


REPO = find_repo_root()
sys.path.insert(0, str(REPO / "analysis"))
import membrane_spectrum as ms
import membrane_resample as mr
import membrane_fluid_surface as mfs
import fluidity as fl

ROOT = Path(os.environ.get("SLIMED_FLAT_RUNS", "~/slimed_runs/flat_tension/matrix")).expanduser()
OUT = ROOT / "analysis"
OUT.mkdir(parents=True, exist_ok=True)
KBT = 4.17
NBLOCK = 8          # blocks behind every error bar
LEVELS = 2          # subdivision levels of the resampler
QDX_MAX = np.pi / 2 # the continuum window: four cells per wavelength
BURN_FRACTION = 0.2 # of the run, discarded before every average

mpl.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "figure.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
def nice_logx(*axes, ticks=(0.05, 0.1, 0.2, 0.3, 0.5, 0.7)):
    from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
    for ax in axes:
        ax.xaxis.set_major_locator(FixedLocator(list(ticks))); ax.xaxis.set_minor_locator(NullLocator())
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))


FLUIDITY_COLORS = {"frozen": "#4c5c68", "nu0.05": "#2a7f9e", "nu0.5": "#d9822b", "nu5": "#b5342e"}
FLUIDITY_LABELS = {"frozen": "frozen", "nu0.05": "ν = 0.05/µs", "nu0.5": "ν = 0.5/µs", "nu5": "ν = 5/µs"}
RESERVOIRS = ["box", "s0global", "s0local", "fixed"]
RESERVOIR_LABELS = {"box": "box scaling", "s0global": "adaptive S0 (global)", "s0local": "adaptive S0 (local)", "fixed": "fixed box"}
RESERVOIR_STYLES = {"box": "-", "s0global": "--", "s0local": ":", "fixed": "-."}
print("runs:", ROOT, "\noutput:", OUT)

---
## 0. The runs

Each run is read whole: the control net of every frame, the connectivity at
every flip snapshot, the flip log, and the reservoir's own log (the box's
strain and tension by term, or the lipid reservoir's relaxed area and mean
tension). The first 20 % of every run is burn-in and enters no average.

In [ ]:
def read_params(path):
    out = {}
    for line in open(path):
        if "=" in line and not line.strip().startswith("#"):
            k, v = line.split("=", 1)
            out[k.strip()] = v.split("#")[0].strip()
    return out


def read_log(path):
    return np.genfromtxt(path, delimiter=",", names=True) if path.exists() else None


def at_frames(table, iterations, column):
    # the logs are written every step or every few steps; sample them at the coordinate frames
    return np.interp(iterations, table["iteration"], table[column])


class FlatRun:
    def __init__(self, directory):
        self.dir = Path(directory)
        self.name = self.dir.name
        p = self.params = read_params(self.dir / "input.params")
        self.dt = float(p["timeStep"]); self.interval = int(p["meshpointOutputInterval"])
        self.dt_frame = self.dt * self.interval
        self.K = float(p["usMembraneStretching"]); self.kc = float(p["kcMembraneBending"])
        self.D = float(p.get("diffConst", 1.0))
        self.flips = p.get("edgeFlipEnabled", "false") == "true"
        rate = float(p.get("edgeFlipAttemptRate", 0)) if self.flips else 0.0
        self.fluidity = "frozen" if not self.flips else {0.05: "nu0.05", 0.5: "nu0.5", 5.0: "nu5"}.get(rate, f"nu{rate:g}")
        if p.get("elasticBoxEnabled", "false") == "true": self.reservoir = "box"
        elif p.get("lipidFlowMode", "off") == "global": self.reservoir = "s0global"
        elif p.get("lipidFlowMode", "off") == "local": self.reservoir = "s0local"
        else: self.reservoir = "fixed"
        self.target = float(p.get("elasticBoxTension" if self.reservoir == "box" else "lipidReservoirTension", 0.0))
        self.label = f"{RESERVOIR_LABELS[self.reservoir]}, {FLUIDITY_LABELS[self.fluidity]}"

        self.run = fl.load_run(str(self.dir), "input", self.interval)
        self.run.free_vertices = fl._free_vertices_from_types(self.run)
        self.coords = self.run.coords
        self.it = self.run.frame_iterations
        self.time = self.it * self.dt
        self.n = len(self.time)
        self.burn = int(BURN_FRACTION * self.n)
        self.lat = ms.lattice_from_params(str(self.dir / "input.params"))

        # The reservoir's reading of the tension, and the box's scale per frame.
        self.scale = np.ones(self.n)
        self.terms = {}
        box = read_log(self.dir / "inputbox.csv"); lipid = read_log(self.dir / "inputlipid.csv")
        if box is not None:
            self.scale = at_frames(box, self.it, "scaleX")
            self.tension = at_frames(box, self.it, "tension_pN_nm")
            for col in box.dtype.names:
                if col.startswith("tension") and col not in ("tension_pN_nm", "tensionX_pN_nm", "tensionY_pN_nm", "tensionZ_pN_nm"):
                    self.terms[col.replace("tension", "").replace("_pN_nm", "")] = at_frames(box, self.it, col)
            self.tension_label = "frame tension, box virial"
        elif lipid is not None:
            self.tension = at_frames(lipid, self.it, "meanTension_pN_nm")
            self.tension_label = "face mean of K (S − S0)/S0"
        else:
            # no reservoir: K (A - A0)/A0 with A the control net's area, A0 its value as built
            area = np.array([self.net_area(k) for k in range(self.n)])
            self.tension = self.K * (area / area[0] - 1.0)
            self.tension_label = "K (A − A0)/A0, control net"
        # accepted flips per microsecond, in the windows between frames
        self.flips_per_us = np.zeros(self.n)
        if self.run.flip_accepted is not None:
            acc = self.run.flip_iteration[self.run.flip_accepted]
            counts, _ = np.histogram(acc, bins=np.append(self.it, self.it[-1] + self.interval))
            self.flips_per_us = counts / self.dt_frame

    def faces_at(self, k):
        usable = [i for i in self.run.face_frames if i <= self.it[k]]
        return self.run.face_frames[max(usable) if usable else min(self.run.face_frames)]

    def net_area(self, k):
        f = self.faces_at(k); f = f[self.run.free_vertices[f].all(axis=1)] if False else f
        P = self.coords[k][f]
        return 0.5 * np.linalg.norm(np.cross(P[:, 1] - P[:, 0], P[:, 2] - P[:, 0]), axis=1).sum() * self.lat.area / self.lat.area  # whole net; ratio is what matters

    def analysed(self):
        return slice(self.burn, self.n)


names = sorted(d.name for d in ROOT.iterdir() if (d / "input.params").exists() and (d / "meshpointinput.csv").exists())
runs = []
for name in names:
    try:
        t0 = time.perf_counter()
        runs.append(FlatRun(ROOT / name))
        r = runs[-1]
        print(f"{name:18s} {r.n:5d} frames to {r.time[-1]:6.1f} µs, scale {r.scale[r.analysed()].mean():.4f}, "
              f"{r.flips_per_us[r.analysed()].mean():7.1f} flips/µs, {time.perf_counter() - t0:4.0f} s")
    except Exception as exc:
        print(f"skipping {name}: {exc}")
runs.sort(key=lambda r: (RESERVOIRS.index(r.reservoir), list(FLUIDITY_COLORS).index(r.fluidity)))
lat = runs[0].lat
A_CELL = lat.area / (lat.nx * lat.ny)
print(lat.describe())
print(f"area per vertex {A_CELL:.2f} nm², target tension {runs[0].target} pN/nm")

---
## 1. Mean tension

Each reservoir controls a different variable and logs it: the box its frame
tension, the in-plane virial over the projected area with the vertex-gas term
$-Nk_BT/A_p$ included; the relaxed-area modes the face mean of
$\sigma_f = K\,(S_f - S_{0,f})/S_{0,f}$; the fixed box has no control and its
$K\,(A - A_0)/A_0$ is simply what the thermal roughness costs. The error bars
are the standard error over eight blocks of the analysed window.

In [ ]:
def block_mean(y, nblock=NBLOCK):
    edges = np.linspace(0, len(y), nblock + 1).astype(int)
    b = np.array([np.nanmean(y[a:c]) for a, c in zip(edges[:-1], edges[1:])])
    return b.mean(), b.std(ddof=1) / np.sqrt(nblock)


fig, axes = plt.subplots(1, len(RESERVOIRS), figsize=(4.2 * len(RESERVOIRS), 3.4), sharey=True)
for ax, res in zip(axes, RESERVOIRS):
    for r in runs:
        if r.reservoir != res: continue
        w = max(1, int(2.0 / r.dt_frame))   # a 2 µs running mean
        ax.plot(r.time, np.convolve(r.tension, np.ones(w) / w, mode="same"), color=FLUIDITY_COLORS[r.fluidity], lw=1.0,
                label=FLUIDITY_LABELS[r.fluidity])
    ax.axhline(runs[0].target, color="k", lw=0.8, ls=":")
    ax.axvspan(0, runs[0].time[runs[0].burn], color="0.92", lw=0)
    ax.set_title(RESERVOIR_LABELS[res], fontsize=10); ax.set_xlabel("time (µs)")
axes[0].set_ylabel("the reservoir's tension (pN/nm), 2 µs mean"); axes[0].legend(fontsize=8)
axes[0].set_ylim(-1.0, 1.5)
plt.tight_layout(); fig.savefig(OUT / "tension_traces.png"); plt.show()

tension_stats = {}
rows = ["| run | reservoir reads | mean (pN/nm) | ± sem | box: bending | area | mesh | vertex gas |", "|---|---|---|---|---|---|---|---|"]
for r in runs:
    m, e = block_mean(r.tension[r.analysed()]); tension_stats[r.name] = (m, e)
    terms = {k: np.nanmean(v[r.analysed()]) for k, v in r.terms.items()}
    cell = lambda key: f"{terms[key]:+.3f}" if key in terms else ""
    rows.append(f"| {r.name} | {r.tension_label} | {m:+.3f} | {e:.3f} | {cell('Bending')} | {cell('AreaConstraint')} | {cell('MeshTerm')} | {cell('VertexGas')} |")
display(Markdown("\n".join(rows)))

---
## 2. The height field on the limit surface

The sheet is read the way the fluid notebook reads it: the limit surface of the
control net, built by Loop subdivision of the mesh's connectivity at every
flip snapshot, is interpolated on a cartesian grid over the tile. Under box
scaling the tile's size changes with the strain, so the grid is scaled with
the box frame by frame and the spectrum is taken in scaled coordinates with
the mean box as the period: the modes keep their index, and $|q|$ is read
off the mean tile.

In [ ]:
net0 = mr.reference_net(str(runs[0].dir / "meshpointinput.csv"), lat)
X0, Y0 = mr.LimitSurface(lat, net0).monge_grid("cartesian")
qx, qy, Q2D = mr.cartesian_q_grid(lat)


def unique_modes(lat_):
    keep = np.zeros((lat_.ny, lat_.nx), bool); seen = set()
    for iy in range(lat_.ny):
        for ix in range(lat_.nx):
            if (iy, ix) == (0, 0) or ((-iy) % lat_.ny, (-ix) % lat_.nx) in seen: continue
            seen.add((iy, ix)); keep[iy, ix] = True
    return keep


KEEP = unique_modes(lat)


def heights(r, frames):
    # the limit surface on the grid scaled with the box (scale 1 without the box)
    cache = {}
    h = np.empty((len(frames),) + X0.shape)
    for j, k in enumerate(frames):
        usable = [i for i in r.run.face_frames if i <= r.it[k]]
        key = max(usable) if usable else min(r.run.face_frames)
        surf = cache.get(key)
        if surf is None:
            surf = cache[key] = mfs.SubdividedSurface(r.run.face_frames[key], r.coords.shape[1], LEVELS)
        h[j] = surf.interpolate(r.coords[k], X0 * r.scale[k], Y0 * r.scale[k])
    return h


for r in runs:
    t0 = time.perf_counter()
    frames = np.arange(r.burn, r.n)
    r.h = heights(r, frames)
    assert np.isfinite(r.h).all(), f"{r.name}: a grid point fell outside the mesh"
    r.h -= r.h.mean(axis=(1, 2), keepdims=True)
    r.h_fft = np.fft.fft2(r.h, axes=(-2, -1))
    r.area_mean = lat.area * r.scale[r.analysed()].mean() ** 2
    r.q = Q2D / r.scale[r.analysed()].mean()
    print(f"{r.name:18s} {len(frames):5d} frames resampled in {time.perf_counter() - t0:4.0f} s, "
          f"r.m.s. height {r.h.std():.3f} nm, {len({max([i for i in r.run.face_frames if i <= it] or [0]) for it in r.it[r.burn:]})} connectivities")

---
## 3. The spatial spectrum

$$\langle |h_q|^2 \rangle = \frac{k_BT}{A\,(\kappa_c q^4 + \sigma q^2)}$$

fitted as $1/(A S) = (\kappa_c/k_BT)\,q^4 + (\sigma/k_BT)\,q^2$ over the
continuum window $|q|\,d \le \pi/2$, one fit per time block, mean and standard
error over the blocks. $\sigma$ here is the tension the *spectrum* reads, which
is the frame tension $\tau$ for the box (section 3.3 of `docs/elastic_box.md`)
and not the stretching tension the relaxed-area modes log.

In [ ]:
def spectrum(r, nblock=NBLOCK):
    n = len(r.h_fft); edges = np.linspace(0, n, nblock + 1).astype(int)
    per = np.array([np.mean(np.abs(r.h_fft[a:b]) ** 2, axis=0) for a, b in zip(edges[:-1], edges[1:])]) / (lat.nx * lat.ny) ** 2
    S, E = per.mean(axis=0), per.std(axis=0, ddof=1) / np.sqrt(nblock)
    qv, Sv, Ev = r.q[KEEP], S[KEEP], E[KEEP]
    o = np.argsort(qv); qv, Sv, Ev = qv[o], Sv[o], Ev[o]
    win = qv * lat.d_face_x * r.scale[r.analysed()].mean() <= QDX_MAX
    fits = []
    for a, b in zip(edges[:-1], edges[1:]):
        Sb = np.mean(np.abs(r.h_fft[a:b]) ** 2, axis=0)[KEEP][o] / (lat.nx * lat.ny) ** 2
        fits.append(ms.fit_kc_sigma(qv[win], Sb[win], r.area_mean, KBT))
    fits = np.array(fits)
    slope = np.polyfit(np.log10(qv[win]), np.log10(Sv[win]), 1)[0]
    return dict(q=qv, S=Sv, E=Ev, win=win, slope=slope, kc=fits[:, 0].mean(), kc_err=fits[:, 0].std(ddof=1) / np.sqrt(nblock),
                sigma=fits[:, 1].mean(), sigma_err=fits[:, 1].std(ddof=1) / np.sqrt(nblock))


for r in runs:
    r.spec = spectrum(r)

fig, axes = plt.subplots(1, len(RESERVOIRS), figsize=(4.2 * len(RESERVOIRS), 3.8), sharey=True)
for ax, res in zip(axes, RESERVOIRS):
    for r in runs:
        if r.reservoir != res: continue
        s = r.spec
        ax.errorbar(s["q"], s["S"], s["E"], fmt="o", ms=3, lw=0.8, color=FLUIDITY_COLORS[r.fluidity], alpha=0.8,
                    label=f"{FLUIDITY_LABELS[r.fluidity]}: κ {s['kc']:.0f}, σ {s['sigma']:+.2f}")
    qq = np.geomspace(0.05, 0.7, 50)
    ax.plot(qq, KBT / (runs[0].area_mean * runs[0].kc * qq ** 4), "k:", lw=1, label=f"kT/(A κ q⁴), κ = {runs[0].kc:.0f}")
    ax.axvline(QDX_MAX / lat.d_face_x, color="0.7", lw=0.8)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("|q| (1/nm)"); ax.set_title(RESERVOIR_LABELS[res], fontsize=10)
    ax.legend(fontsize=7)
axes[0].set_ylabel("⟨|h_q|²⟩ (nm²)"); nice_logx(*axes)
plt.tight_layout(); fig.savefig(OUT / "spectra.png"); plt.show()

rows = ["| run | modes in window | slope | κ_c (pN·nm) | ± | σ spectrum (pN/nm) | ± | reservoir's mean σ |", "|---|---|---|---|---|---|---|---|"]
for r in runs:
    s = r.spec
    rows.append(f"| {r.name} | {s['win'].sum()} | {s['slope']:+.2f} | {s['kc']:.1f} | {s['kc_err']:.1f} | {s['sigma']:+.3f} | {s['sigma_err']:.3f} | {tension_stats[r.name][0]:+.3f} |")
display(Markdown("\n".join(rows)))
print(f"input κ_c = {runs[0].kc} pN·nm")

---
## 4. The temporal spectrum

Each mode's autocorrelation $C_q(t) = \langle h_q(t)\,h_q^*(0)\rangle / \langle|h_q|^2\rangle$
is integrated to its $1/e$ crossing to give a relaxation rate $\Gamma(q)$
(`membrane_spectrum.relaxation_rate`, with its two guards: a mode slower than
5 % of the record or faster than two frames is not reported). The vertices are
free-draining Brownian particles, so the prediction is

$$\Gamma(q) = \frac{D\,a}{k_BT}\,\bigl(\kappa_c q^4 + \sigma q^2\bigr),$$

with $a$ the area per vertex: no hydrodynamics, $\Gamma \propto q^4$ at zero
tension. Flips are Monte Carlo moves and carry no clock, so if the sheet's
dynamics is unchanged by them $\Gamma(q)$ should lie on the same curve for
every fluidity; the relaxed-area modes add a slow field of their own.

In [ ]:
def rates(r):
    acf = ms.mode_autocorrelation(r.h_fft)
    G = np.full(acf.shape[1:], np.nan)
    for iy in range(acf.shape[1]):
        for ix in range(acf.shape[2]):
            if KEEP[iy, ix]:
                G[iy, ix] = ms.relaxation_rate(acf[:, iy, ix], r.dt_frame, n_frames=len(r.h_fft))
    return acf, G


def shells(q, G, n=10):
    ok = np.isfinite(G) & KEEP
    edges = np.geomspace(q[ok].min() * 0.99, q[ok].max() * 1.01, n + 1)
    out = []
    for a, b in zip(edges[:-1], edges[1:]):
        m = ok & (q >= a) & (q < b)
        if m.sum() >= 2:
            out.append((np.exp(np.log(q[m]).mean()), G[m].mean(), G[m].std(ddof=1) / np.sqrt(m.sum()), m.sum()))
    return np.array(out)


for r in runs:
    r.acf, r.G = rates(r)
    r.shells = shells(r.q, r.G)
    ok = np.isfinite(r.G) & KEEP & (r.q * lat.d_face_x <= QDX_MAX)
    pred = r.D * A_CELL / KBT * (r.kc * r.q ** 4 + max(r.spec["sigma"], 0.0) * r.q ** 2)
    r.rate_ratio = (np.nanmean(r.G[ok] / pred[ok]), np.nanstd(r.G[ok] / pred[ok], ddof=1) / np.sqrt(ok.sum())) if ok.sum() > 1 else (np.nan, np.nan)
    r.rate_slope = np.polyfit(np.log(r.q[ok]), np.log(r.G[ok]), 1)[0] if ok.sum() > 3 else np.nan

fig, axes = plt.subplots(1, len(RESERVOIRS), figsize=(4.2 * len(RESERVOIRS), 3.8), sharey=True)
for ax, res in zip(axes, RESERVOIRS):
    for r in runs:
        if r.reservoir != res or len(r.shells) == 0: continue
        ax.errorbar(r.shells[:, 0], r.shells[:, 1], r.shells[:, 2], fmt="o-", ms=3, lw=0.9, color=FLUIDITY_COLORS[r.fluidity],
                    label=f"{FLUIDITY_LABELS[r.fluidity]}: Γ/Γ_pred {r.rate_ratio[0]:.2f}, slope {r.rate_slope:.1f}")
    qq = np.geomspace(0.06, 0.7, 50)
    ax.plot(qq, runs[0].D * A_CELL / KBT * runs[0].kc * qq ** 4, "k:", lw=1, label="(D a/kT) κ q⁴")
    ax.axvline(QDX_MAX / lat.d_face_x, color="0.7", lw=0.8)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("|q| (1/nm)"); ax.set_title(RESERVOIR_LABELS[res], fontsize=10)
    ax.legend(fontsize=7)
axes[0].set_ylabel("Γ(q) (1/µs)"); nice_logx(*axes)
plt.tight_layout(); fig.savefig(OUT / "relaxation_rates.png"); plt.show()

# a few modes' autocorrelations, one run per reservoir at nu = 0.5
fig, axes = plt.subplots(1, len(RESERVOIRS), figsize=(4.2 * len(RESERVOIRS), 3.2), sharey=True)
for ax, res in zip(axes, RESERVOIRS):
    pick = [r for r in runs if r.reservoir == res and r.fluidity == "nu0.5"] or [r for r in runs if r.reservoir == res]
    if not pick: continue
    r = pick[0]
    for (iy, ix), c in zip(((0, 2), (0, 4), (2, 4), (0, 8)), ("#1f4e79", "#2a7f9e", "#d9822b", "#b5342e")):
        lag = np.arange(len(r.acf)) * r.dt_frame
        m = lag <= 40
        ax.plot(lag[m], r.acf[m, iy, ix], color=c, lw=1, label=f"|q| = {r.q[iy, ix]:.2f}")
        g = r.G[iy, ix]
        if np.isfinite(g): ax.plot(lag[m], np.exp(-g * lag[m]), color=c, lw=0.6, ls="--")
    ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("lag (µs)"); ax.set_title(r.label, fontsize=9); ax.legend(fontsize=7)
axes[0].set_ylabel("C_q(t)")
plt.tight_layout(); fig.savefig(OUT / "autocorrelations.png"); plt.show()

---
## 5. The numbers

One row per run; `stats.md` beside the figures. $\Gamma/\Gamma_{\rm pred}$ is
the mean over the resolved modes of the continuum window of the measured rate
over the free-draining prediction with the input $\kappa_c$ and the fitted
$\sigma$ (floored at zero); the slope is $d\ln\Gamma/d\ln q$ over the same modes.

In [ ]:
rows = ["| run | reservoir | fluidity | flips/µs | box scale | σ reservoir (pN/nm) | σ spectrum | κ_c spectrum (pN·nm) | h r.m.s. (nm) | Γ/Γ_pred | Γ slope | resolved modes |",
        "|---|---|---|---|---|---|---|---|---|---|---|---|"]
for r in runs:
    m, e = tension_stats[r.name]; s = r.spec
    nres = int((np.isfinite(r.G) & KEEP & (r.q * lat.d_face_x <= QDX_MAX)).sum())
    rows.append(f"| {r.name} | {RESERVOIR_LABELS[r.reservoir]} | {FLUIDITY_LABELS[r.fluidity]} | {r.flips_per_us[r.analysed()].mean():.0f} | "
                f"{r.scale[r.analysed()].mean():.4f} | {m:+.3f} ± {e:.3f} | {s['sigma']:+.3f} ± {s['sigma_err']:.3f} | {s['kc']:.1f} ± {s['kc_err']:.1f} | "
                f"{r.h.std():.3f} | {r.rate_ratio[0]:.2f} ± {r.rate_ratio[1]:.2f} | {r.rate_slope:.2f} | {nres} |")
table = "\n".join(rows)
(OUT / "stats.md").write_text(table + "\n")
display(Markdown(table))

---
## 6. Animations

The height field of every run on the Monge grid, one GIF per run and one for
the whole matrix, written to `<root>/analysis` (not tracked). Set `MAKE_GIFS`
to `False` to skip them.

In [ ]:
MAKE_GIFS = True
N_FRAMES, FPS = 120, 12
if MAKE_GIFS:
    vmax = max(np.percentile(np.abs(r.h), 99.5) for r in runs)
    ext = lambda r, k: (0, lat.lx * r.scale[k], 0, lat.ly * r.scale[k])

    def matrix_gif(path):
        res = [k for k in RESERVOIRS if any(r.reservoir == k for r in runs)]
        flu = [k for k in FLUIDITY_COLORS if any(r.fluidity == k for r in runs)]
        fig, axes = plt.subplots(len(res), len(flu), figsize=(3.0 * len(flu), 2.9 * len(res)), squeeze=False)
        ims, stamp = {}, fig.text(0.5, 0.985, "", ha="center", fontsize=10)
        for i, rs in enumerate(res):
            for j, fl_ in enumerate(flu):
                ax = axes[i, j]; ax.set_xticks([]); ax.set_yticks([])
                m = [r for r in runs if r.reservoir == rs and r.fluidity == fl_]
                if not m: ax.set_facecolor("0.94"); ax.text(0.5, 0.5, "no run", ha="center", va="center", transform=ax.transAxes); continue
                r = m[0]
                ims[r.name] = ax.imshow(r.h[0], origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax, extent=ext(r, r.burn), animated=True)
                ax.set_title(r.label, fontsize=8)
        n = min(len(r.h) for r in runs); picked = np.linspace(0, n - 1, N_FRAMES).astype(int)
        fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.02, hspace=0.25, wspace=0.08)
        fig.colorbar(list(ims.values())[0], ax=axes.ravel().tolist(), shrink=0.5, pad=0.01).set_label("height (nm)")
        def update(k):
            for r in runs:
                if r.name in ims: ims[r.name].set_array(r.h[k]); ims[r.name].set_extent(ext(r, r.burn + k))
            stamp.set_text(f"t = {runs[0].time[runs[0].burn + k]:.1f} µs")
            return list(ims.values()) + [stamp]
        animation.FuncAnimation(fig, update, frames=picked, blit=False).save(path, writer=animation.PillowWriter(fps=FPS), dpi=90)
        plt.close(fig)

    def run_gif(r, path):
        fig, (ax, ax2) = plt.subplots(1, 2, figsize=(8.4, 3.6), gridspec_kw=dict(width_ratios=(1.1, 1)))
        im = ax.imshow(r.h[0], origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax, extent=ext(r, r.burn))
        fig.colorbar(im, ax=ax, shrink=0.8).set_label("height (nm)"); ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
        ax2.plot(r.time, r.tension, color="0.75", lw=0.6); ax2.axhline(r.target, color="k", lw=0.6, ls=":")
        cur = ax2.axvline(r.time[r.burn], color=FLUIDITY_COLORS[r.fluidity]); ax2.set_xlabel("time (µs)"); ax2.set_ylabel(r.tension_label + " (pN/nm)")
        ax2.set_ylim(-1.5, 2.0); title = fig.suptitle("", fontsize=10)
        picked = np.linspace(0, len(r.h) - 1, N_FRAMES).astype(int)
        def update(k):
            im.set_array(r.h[k]); im.set_extent(ext(r, r.burn + k)); cur.set_xdata([r.time[r.burn + k]] * 2)
            title.set_text(f"{r.label}: t = {r.time[r.burn + k]:.1f} µs, box scale {r.scale[r.burn + k]:.3f}")
            return im, cur, title
        fig.tight_layout()
        animation.FuncAnimation(fig, update, frames=picked, blit=False).save(path, writer=animation.PillowWriter(fps=FPS), dpi=90)
        plt.close(fig)

    t0 = time.perf_counter()
    matrix_gif(OUT / "matrix.gif")
    for r in runs:
        run_gif(r, OUT / f"anim_{r.name}.gif")
    print(f"{len(runs) + 1} animations in {time.perf_counter() - t0:.0f} s")
    display(Image(filename=str(OUT / "matrix.gif")))